# SisaKulkas - Training v2 (lebih sedikit objek terlewat)

Model sebelumnya: YOLOv8**n** (varian terkecil), 640 px, 50 epoch, recall 0,906 di data validasi. Notebook ini melakukan tiga hal:

1. **Diagnosa**: tabel recall per kelas untuk melihat bahan mana yang paling sering terlewat.
2. **Training ulang** dengan model lebih besar (YOLOv8s), epoch lebih banyak, dan augmentasi yang lebih sesuai foto kulkas.
3. **Ekspor ONNX** untuk aplikasi web, plus `config.json` otomatis (ukuran input dan nama kelas).

**Sebelum mulai:** Runtime > Change runtime type > **T4 GPU**. Simpan API key Roboflow (yang baru) di **Secrets** dengan nama `ROBOFLOW_API_KEY`, aktifkan *Notebook access*.

In [ ]:
!pip install -q roboflow ultralytics

## 1. Pengaturan
Salin `WORKSPACE`, `PROJECT`, dan `VERSION` dari tombol **Download Dataset** di Roboflow (pilih format YOLOv8).

In [ ]:
WORKSPACE = "veraluckyray282002-gmail-com"
PROJECT   = "laststretch"
VERSION   = 1

MODEL  = "yolov8s.pt"  # yolov8n.pt (tercepat) | yolov8s.pt (disarankan) | yolov8m.pt (lebih akurat, lebih berat)
IMGSZ  = 640           # 640 cepat. 800 membantu objek kecil, tapi deteksi di web jadi lebih lambat
EPOCHS = 100
BATCH  = 16            # jika muncul "CUDA out of memory", turunkan ke 8

## 2. Unduh dataset

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print("Lokasi:", dataset.location)

## 3. Rapikan data.yaml
Mencari folder validasi yang ada. Jika tidak ada, 15% data train dipisah sebagai validasi.

In [ ]:
import os, glob, random, shutil, yaml

loc = dataset.location
has = lambda s: os.path.isdir(f"{loc}/{s}/images")

VAL = next((s for s in ["valid", "val"] if has(s)), None)
if VAL is None:
    os.makedirs(f"{loc}/valid/images", exist_ok=True)
    os.makedirs(f"{loc}/valid/labels", exist_ok=True)
    imgs = sorted(glob.glob(f"{loc}/train/images/*"))
    random.seed(0); random.shuffle(imgs)
    for img in imgs[:max(1, int(0.15 * len(imgs)))]:
        base = os.path.splitext(os.path.basename(img))[0]
        shutil.move(img, f"{loc}/valid/images/")
        lbl = f"{loc}/train/labels/{base}.txt"
        if os.path.exists(lbl):
            shutil.move(lbl, f"{loc}/valid/labels/")
    VAL = "valid"

with open(f"{loc}/data.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = loc
cfg["train"] = "train/images"
cfg["val"] = f"{VAL}/images"
cfg.pop("test", None)
if has("test"):
    cfg["test"] = "test/images"
with open(f"{loc}/data.yaml", "w") as f:
    yaml.safe_dump(cfg, f)

DATA_YAML = f"{loc}/data.yaml"
names = cfg["names"]
names = [names[i] for i in sorted(names)] if isinstance(names, dict) else names
print("Validasi:", VAL, "| jumlah kelas:", len(names))

## 4. Periksa data
Kelas dengan data sedikit hampir pasti sering terlewat. Objek yang sangat kecil di foto juga sulit dideteksi.

In [ ]:
from collections import Counter
import pandas as pd

def count_split(split):
    cnt, small, total = Counter(), 0, 0
    for lf in glob.glob(f"{loc}/{split}/labels/*.txt"):
        for line in open(lf):
            p = line.split()
            if len(p) >= 5:
                cnt[int(p[0])] += 1
                total += 1
                if float(p[3]) * float(p[4]) < 0.01:   # kurang dari 1% luas gambar
                    small += 1
    return cnt, small, total

tr, small_tr, tot_tr = count_split("train")
va, _, _ = count_split(VAL)
df = pd.DataFrame({"kelas": names,
                   "objek_train": [tr[i] for i in range(len(names))],
                   "objek_valid": [va[i] for i in range(len(names))]})
df["catatan"] = df["objek_train"].apply(lambda n: "DATA SEDIKIT (<100)" if n < 100 else "")
display(df.sort_values("objek_train").reset_index(drop=True))
print(f"Gambar train: {len(glob.glob(loc + '/train/images/*'))}, objek kecil (<1% luas gambar): {small_tr}/{tot_tr} ({100*small_tr/max(tot_tr,1):.0f}%)")

## 5. (Opsional) Diagnosa model lama
Unggah `best.pt` lama ke Colab (ikon folder di kiri). Tabel urut dari **recall terendah**: itulah bahan yang paling sering terlewat.

In [ ]:
from ultralytics import YOLO

def per_class(metrics, title):
    rows = []
    for j, ci in enumerate(metrics.box.ap_class_index):
        rows.append({"kelas": metrics.names[int(ci)], "precision": metrics.box.p[j],
                     "recall": metrics.box.r[j], "AP50": metrics.box.ap50[j], "AP50-95": metrics.box.ap[j]})
    out = pd.DataFrame(rows).sort_values("recall").round(3).reset_index(drop=True)
    print(title)
    display(out)
    return out

OLD = "best.pt"
if os.path.exists(OLD):
    m_old = YOLO(OLD).val(data=DATA_YAML, imgsz=640, verbose=False)
    per_class(m_old, f"Model lama (mAP50={m_old.box.map50:.3f}, recall={m_old.box.mr:.3f})")
else:
    print("best.pt lama tidak ditemukan, bagian ini dilewati.")

## 6. Latih model baru
Perubahan utama: model lebih besar, 100 epoch dengan *early stopping* (berhenti jika 30 epoch tidak membaik), augmentasi lebih beragam, dan mosaic dimatikan di 15 epoch terakhir.

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=30,
    cos_lr=True,
    close_mosaic=15,
    mosaic=1.0,
    mixup=0.1,
    scale=0.6,
    degrees=5,
    translate=0.1,
    fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
    project="sisakulkas",
    name="v2",
    seed=0,
)
BEST = str(model.trainer.best)
print("Model terbaik:", BEST)

## 7. Evaluasi model baru

In [ ]:
best = YOLO(BEST)
m_new = best.val(data=DATA_YAML, imgsz=IMGSZ, verbose=False)
print(f"mAP50={m_new.box.map50:.3f}  mAP50-95={m_new.box.map:.3f}  precision={m_new.box.mp:.3f}  recall={m_new.box.mr:.3f}")
per_class(m_new, "Model baru per kelas (urut dari recall terendah)")

In [ ]:
from IPython.display import Image as IPImage, display
cm = f"{model.trainer.save_dir}/confusion_matrix_normalized.png"
if os.path.exists(cm):
    display(IPImage(cm))

## 8. Lihat contoh prediksi (ambang 25%)

In [ ]:
val_imgs = sorted(glob.glob(f"{loc}/{VAL}/images/*"))[:6]
preds = best.predict(val_imgs, conf=0.25, imgsz=IMGSZ, save=True, project="sisakulkas", name="contoh", exist_ok=True, verbose=False)
for p in preds:
    display(IPImage(os.path.join(p.save_dir, os.path.basename(p.path))))

## 9. Ekspor untuk aplikasi web
Menghasilkan `best.onnx` dan `config.json` (ukuran input + nama kelas). Ganti kedua file di folder aplikasi web: `model/best.onnx` dan `config.json` di folder utama.

In [ ]:
import json
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True)
class_names = [best.names[i] for i in sorted(best.names)]
with open("config.json", "w", encoding="utf-8") as f:
    json.dump({"imgsz": IMGSZ, "names": class_names}, f, ensure_ascii=False)
shutil.copy(onnx_path, "best.onnx")

from google.colab import files
files.download("best.onnx")
files.download("config.json")
files.download(BEST)   # best.pt baru, simpan sebagai cadangan

## Jika masih ada objek yang terlewat
- **Lihat tabel recall per kelas.** Kelas dengan recall rendah biasanya datanya sedikit atau tampilannya mirip kelas lain (lihat confusion matrix).
- **Tambah foto asli** dari kulkas/dapur nyata untuk kelas yang lemah, beri label di Roboflow, lalu training ulang. Ini perbaikan paling efektif, karena data online sering berbeda dari foto kulkas yang berantakan.
- **Coba `IMGSZ = 800`** jika banyak objek kecil, lalu ekspor ulang (config.json otomatis ikut berubah). Deteksi di web jadi lebih lambat.
- **Coba `yolov8m.pt`** jika GPU dan waktu cukup.
- Di aplikasi web, turunkan **ambang keyakinan** di Pengaturan untuk menampilkan lebih banyak objek.